# Airbnb Paris – Exp 4: ConTextTab vs. TabPFN-Klassifikation
- Binäre Klassifikation des ICC-Labels `is_top_rating` (Inlier = rating == 5 → 1, Outlier = rating <= 3 → 0)
- Gemeinsames balanciertes Subset (1:4): Train 120/480 (600 Kontext), Test 30/120 (seed 42)
- ConTextTab: numerisch + Freitext (nativ); TabPFN: nur numerisch
- Pro Modell: Average Precision, AUC-ROC, Classification Report; Logging nach MLflow

In [ ]:
import torch  # vor sap_rpt_oss laden (TORCH_LIBRARY-Doppelregistrierung vermeiden)
import time
import numpy as np
import pandas as pd
import mlflow
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, precision_recall_curve, auc as sk_auc
from tabpfn import TabPFNClassifier
from sap_rpt_oss import SAP_RPT_OSS_Classifier

## Setup & MLflow
- `.env` laden; Tracking nach `../../mlruns`, Experiment `airbnb_paris_experiment_4`

In [2]:
load_dotenv()
SEED = 42
mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_4")

/home/debian/foundational_tabular_od/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///home/debian/TFM_master_thesis/airbnb_notebooks/exp4/../../mlruns/902320835569643670', creation_time=1781194963834, experiment_id='902320835569643670', last_update_time=1781194963834, lifecycle_stage='active', name='airbnb_paris_experiment_4', tags={}, trace_location=None, workspace='default'>

## Daten laden
- `cleaned_text_airbnb_paris.csv` (numerisch/encoded + 4 Freitext, `row_id`-Index); leere Texte als ""
- `X` = numerisch + Freitext (ConTextTab); `Xnum` = nur numerisch (TabPFN)

In [3]:
TEXT_COLS = ["name", "description", "neighborhood_overview", "host_about"]
LABEL = "is_top_rating"

df = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False).set_index("row_id")
X = df.drop(columns=[LABEL])      # numerisch + Freitext
Xnum = X.drop(columns=TEXT_COLS)   # nur numerisch
y = df[LABEL]
print(df.shape, "| Verteilung:", y.value_counts().to_dict())

(18350, 45) | Verteilung: {1: 17629, 0: 721}


## Gemeinsames Subsampling (1:4)
- Outlier = Minderheitsklasse (`is_top_rating == 0`, rating <= 3); Train 120/480 (600 Kontext), Test 30/120 (disjunkt, seed 42)
- Identische Zeilen für beide Modelle → fairer Vergleich

In [4]:
outlier_label = y.value_counts().idxmin()
inlier_label = y.value_counts().idxmax()

# --- 1:4-Verhältnis (auskommentiert) ---
# N_TRAIN_OUT, N_TRAIN_IN = 120, 480   # 600 Kontext, 1:4
# N_TEST_OUT, N_TEST_IN = 30, 120      # 150 Test, 1:4
# rng = np.random.RandomState(SEED)
# idx_out = df.index[y == outlier_label].to_numpy()
# idx_in = df.index[y == inlier_label].to_numpy()
# rng.shuffle(idx_out)
# rng.shuffle(idx_in)
# train_idx = np.concatenate([idx_out[:N_TRAIN_OUT], idx_in[:N_TRAIN_IN]])
# test_idx = np.concatenate([idx_out[N_TRAIN_OUT:N_TRAIN_OUT + N_TEST_OUT],
#                            idx_in[N_TRAIN_IN:N_TRAIN_IN + N_TEST_IN]])
# rng.shuffle(train_idx)
# rng.shuffle(test_idx)

# natürliche Outlier-Verteilung: stratifizierter Split; großer Test für stabile AP
# train = Kontext (< TabPFN-Limit ~4000), test groß genug für genügend Outlier
train_idx, test_idx = train_test_split(df.index.to_numpy(), train_size=2000, test_size=3000,
                                       stratify=y.values, random_state=SEED)

y_train, y_test = y.loc[train_idx], y.loc[test_idx]
y_bin = (y_test == outlier_label).astype(int)                       # binär für AP/AUROC
outlier_col = sorted(np.unique(y_train).tolist()).index(outlier_label)  # P(Outlier)-Spalte
print(f"Train {len(train_idx)} (Outlier {int((y_train == outlier_label).sum())}) | "
      f"Test {len(test_idx)} (Outlier {int((y_test == outlier_label).sum())})")

Train 2000 (Outlier 79) | Test 3000 (Outlier 118)


## ConTextTab (numerisch + Freitext)
- SAP-rpt-1-oss, max_context_size=8192, bagging=8; Score = P(Outlier)

In [ ]:
clf = SAP_RPT_OSS_Classifier(max_context_size=8192, bagging=8)
t0 = time.time()
clf.fit(X.loc[train_idx], y_train)
proba = clf.predict_proba(X.loc[test_idx])[:, outlier_col]
pred = clf.predict(X.loc[test_idx])
runtime = time.time() - t0

ap = average_precision_score(y_bin, proba)
auc = roc_auc_score(y_bin, proba)
prec, rec, _ = precision_recall_curve(y_bin, proba)
aucpr = sk_auc(rec, prec)
print(f"ConTextTab – AP={ap:.4f}  AUCPR={aucpr:.4f}  AUC-ROC={auc:.4f}  t={runtime:.1f}s")
print(classification_report(y_test, pred, digits=4, zero_division=0))

with mlflow.start_run(run_name="contexttab"):
    mlflow.log_params({"n_train": len(train_idx), "n_test": len(test_idx), "n_features": X.shape[1],
                       "max_context_size": 8192, "bagging": 8, "random_state": SEED})
    mlflow.log_metrics({"average_precision": float(ap), "auc_roc": float(auc), "aucpr": float(aucpr), "runtime_s": round(runtime, 2)})

## TabPFN (nur numerisch)
- Freitexte gedroppt; Score = P(Outlier)

In [ ]:
tab = TabPFNClassifier()
t0 = time.time()
tab.fit(Xnum.loc[train_idx].values, y_train.values)
proba = tab.predict_proba(Xnum.loc[test_idx].values)[:, outlier_col]
pred = tab.predict(Xnum.loc[test_idx].values)
runtime = time.time() - t0

ap = average_precision_score(y_bin, proba)
auc = roc_auc_score(y_bin, proba)
prec, rec, _ = precision_recall_curve(y_bin, proba)
aucpr = sk_auc(rec, prec)
print(f"TabPFN – AP={ap:.4f}  AUCPR={aucpr:.4f}  AUC-ROC={auc:.4f}  t={runtime:.1f}s")
print(classification_report(y_test, pred, digits=4, zero_division=0))

with mlflow.start_run(run_name="tabpfn_classification"):
    mlflow.log_params({"n_train": len(train_idx), "n_test": len(test_idx), "n_features": Xnum.shape[1],
                       "random_state": SEED})
    mlflow.log_metrics({"average_precision": float(ap), "auc_roc": float(auc), "aucpr": float(aucpr), "runtime_s": round(runtime, 2)})